# Bus Booking Analytics - ETL Pipeline

This notebook implements the complete **ETL (Extract, Transform, Load)** pipeline as described in the project documentation.
It covers:
1. **Extract**: Loading raw dirty CSV datasets (`bus_booking_raw.csv`, `customers.csv`, `buses.csv`, `routes.csv`) from the local directories.
2. **Transform & Validate**:
   - Flexible Date Parsing using `format='mixed'` to resolve inconsistent date formats.
   - Deduplication (based on `Booking_ID`)
   - Handling Null Values (e.g. dropping bookings with null fares, filling missing customer phone numbers with 'Unknown')
   - Data Standardization (dates formatted to YYYY-MM-DD, proper title casing for names and routes)
   - Data Consistency Checks (Travel Date >= Booking Date, Fare > 0, Seat Number within physical Bus Capacity)
   - Referential Integrity checks between all foreign keys.
3. **Load**: Loading validated data directly into **MySQL** database (`bus_booking_analytics`).

### Step 1: Extract (Load Raw CSVs)

In [1]:
import pandas as pd
import os
import sys

# Import etl_pipeline module from current directory
import etl_pipeline

data_dir = '../data'
bookings, customers, buses, routes = etl_pipeline.load_csv_data(data_dir)
print(f"Raw Bookings count: {len(bookings)}")
print(f"Raw Customers count: {len(customers)}")

Loading CSV files...
Loading raw bookings from ../data\bus_booking_raw.csv
Raw Bookings count: 11631
Raw Customers count: 301


### Step 2: Transform, Clean & Validate
We run the validation logic that details formatting repairs, drop counts, and deduplications.

In [2]:
bookings_clean, customers_clean, buses_clean, routes_clean = etl_pipeline.transform_and_validate(
    bookings, customers, buses, routes
)

--- ETL CLEANING & TRANSFORMATION REPORT ---
Deduplication: Removed 553 duplicate booking records.
Null Handling: Removed 218 booking records with missing Fare_Amount.
Null Handling: Filled 21 missing customer phone numbers with 'Unknown'.
Null Handling/Standardization: Cleansed Gender field, filling 9 missing values.
Null Handling/Outliers: Cleansed Age field, filling 16 nulls and correcting 30 outliers to median (48).
Constraint Check: Removed 213 bookings where Travel_Date was before Booking_Date.
Constraint Check: Removed 160 bookings with non-positive Fare_Amount (e.g. negative values).
Capacity Check: Removed 106 bookings where Seat_Number exceeded Bus Capacity.

--- SUMMARY OF PROCESS ---
Bookings: 11631 raw rows -> 10381 clean rows (Removed 1250 rows).
Customers: 301 raw rows -> 301 clean rows.
--------------------------------------------


### Step 3: Load into MySQL Database
Enter your MySQL connection credentials below to load the clean datasets directly into the database.

In [3]:
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'root',  
    'database': 'bus_booking_analytics'
}

etl_pipeline.load_to_mysql(bookings_clean, customers_clean, buses_clean, routes_clean, mysql_config)

Connecting to MySQL...
Successfully loaded datasets into MySQL database.


### Step 4: Verification Queries via SQL
Let's run queries on our cleaned dataset in the MySQL database to generate Key Business Metrics.

In [4]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import mysql.connector

conn = mysql.connector.connect(
    host=mysql_config['host'],
    user=mysql_config['user'],
    password=mysql_config['password'],
    database=mysql_config['database']
)

print("--- 1. KEY METRICS SUMMARY ---")
metrics_query = """
SELECT 
    COUNT(Booking_ID) as Total_Bookings,
    ROUND(SUM(Fare_Amount), 2) as Total_Revenue,
    ROUND(AVG(Fare_Amount), 2) as Avg_Fare
FROM Bookings
"""
print(pd.read_sql_query(metrics_query, conn))

print("\n--- 2. TOP 5 ROUTES BY REVENUE ---")
routes_query = """
SELECT 
    r.Source, 
    r.Destination, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Route_Revenue
FROM Bookings b
JOIN Routes r ON b.Route_ID = r.Route_ID
GROUP BY r.Route_ID
ORDER BY Route_Revenue DESC
LIMIT 5;
"""
print(pd.read_sql_query(routes_query, conn))

print("\n--- 3. REVENUE BY BUS TYPE ---")
bus_query = """
SELECT 
    bus.Bus_Type, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Revenue
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY bus.Bus_Type
ORDER BY Revenue DESC;
"""
print(pd.read_sql_query(bus_query, conn))

print("\n--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---")
occupancy_query = """
SELECT 
    b.Bus_ID,
    bus.Bus_Number,
    bus.Capacity,
    COUNT(b.Booking_ID) as Bookings_Made,
    ROUND((COUNT(b.Booking_ID) * 100.0) / (bus.Capacity * 20.0), 2) as Estimated_Occupancy_Percentage
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY b.Bus_ID
ORDER BY Estimated_Occupancy_Percentage DESC;
"""
print(pd.read_sql_query(occupancy_query, conn))

conn.close()

--- 1. KEY METRICS SUMMARY ---
   Total_Bookings  Total_Revenue  Avg_Fare
0           10381      8899496.6    857.29

--- 2. TOP 5 ROUTES BY REVENUE ---
      Source Destination  Booking_Count  Route_Revenue
0    Chennai   Hyderabad           1053     1316392.85
1  Bangalore   Hyderabad            993     1153940.39
2       Pune         Goa           1020      919270.83
3     Mumbai   Ahmedabad            891      910550.57
4  Bangalore     Chennai           1107      840612.96

--- 3. REVENUE BY BUS TYPE ---
         Bus_Type  Booking_Count     Revenue
0  Non-AC Sleeper           3841  3380698.02
1      AC Sleeper           3507  3340371.27
2       AC Seater           3033  2178427.31

--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---
    Bus_ID    Bus_Number  Capacity  Bookings_Made  \
0      205  GJ-29-H-5557        50           1373   
1      207  HR-36-C-4527        40           1089   
2      201  TS-18-B-9935        40           1042   
3      206  KA-98-C-7924        40           1037

### Step 5: Run Automated Unit Tests (pytest)
We run `pytest` directly on our script to verify that our data cleansing, validation rules, and capacity limits are correctly tested.

In [5]:
!python -m pytest etl_pipeline.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.4, pytest-9.1.1, pluggy-1.6.0 -- C:\Python312\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\user\Desktop\Revature Phase 2\Bus Booking Analytics System\Bus Booking Analytics System (PowerBI)\etl_pipeline
collecting ... collected 5 items

etl_pipeline.py::test_transform_and_validate_dates PASSED                [ 20%]
etl_pipeline.py::test_transform_and_validate_deduplication PASSED        [ 40%]
etl_pipeline.py::test_transform_and_validate_nulls_and_outliers PASSED   [ 60%]
etl_pipeline.py::test_transform_and_validate_constraints PASSED          [ 80%]
etl_pipeline.py::test_transform_and_validate_capacity_check PASSED       [100%]

============================== 5 passed in 1.33s ==============================
